# Six-Model Surface NBO-OH Analysis

Workflow template for visualizing regenerated stable surface NBO-OH trajectories and last-frame-window O-H statistics for SevenNet-0, MatterSim, UPET, MACE-MH, UMA, and NequIP-OAM-L. Run `scripts/run_oh_stable_all_models.sh` before executing the analysis cells.


In [ ]:

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

ROOT = Path.cwd()
ANALYSIS = ROOT / "analysis_oh_stable"
summary_path = ANALYSIS / "oh_summary.csv"
pair_path = ANALYSIS / "all_oh_pair_summary.csv"
time_path = ANALYSIS / "all_oh_timeseries.csv"
metadata_path = ROOT / "structures" / "hydroxylated_stable" / "oh_site_metadata.json"
perf_summary_path = ROOT / "results" / "computational_performance" / "new_models_oh_stable_memory_timing_summary.csv"

MODEL_ORDER = ["sevennet", "mattersim", "upet", "mace_mh", "uma", "nequip_oam_l"]
MODEL_LABELS = {
    "sevennet": "SevenNet-0",
    "mattersim": "MatterSim",
    "upet": "UPET",
    "mace_mh": "MACE-MH",
    "uma": "UMA",
    "nequip_oam_l": "NequIP-OAM-L",
}
MODEL_COLORS = {
    "sevennet": "#1f77b4",
    "mattersim": "#2ca02c",
    "upet": "#ff7f0e",
    "mace_mh": "#d62728",
    "uma": "#9467bd",
    "nequip_oam_l": "#17becf",
}
MODEL_LS = {"sevennet": "-", "mattersim": "--", "upet": "-.", "mace_mh": ":", "uma": "-", "nequip_oam_l": "--"}
MINERAL_LABELS = {
    "forsterite": "Mg₂SiO₄",
    "fayalite": "Fe₂SiO₄",
    "ilmenite": "TiFeO₃",
    "anorthite": "CaAl₂Si₂O₈",
}
# plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})


## Hydroxylated Surface Sites

In [ ]:

metadata = json.loads(metadata_path.read_text())
site_rows = []
for structure, meta in metadata.items():
    mineral = structure.replace("_1layer_stable_surface_NBO_H", "")
    site_rows.append({
        "structure": structure,
        "mineral": mineral,
        "label": MINERAL_LABELS.get(mineral, mineral),
        "top_surface_o": meta["n_top_surface_o"],
        "surface_nbo_h": meta["n_surface_nbo"],
        "natoms_before_h": meta["natoms_before_h"],
        "natoms_after_h": meta["natoms_after_h"],
        "surface_depth_a": meta["surface_depth_a"],
        "site_rule": meta.get("site_rule"),
        "source_structure": meta.get("source_structure"),
    })
sites = pd.DataFrame(site_rows).sort_values("mineral")
sites


## Last-500-Frame Summary

In [ ]:

summary = pd.read_csv(summary_path)
summary["mineral"] = summary["structure"].str.replace("_1layer_stable_surface_NBO_H", "", regex=False)
summary["mineral_label"] = summary["mineral"].map(MINERAL_LABELS)
summary["model_label"] = summary["model"].map(MODEL_LABELS)
summary["model"] = pd.Categorical(summary["model"], categories=MODEL_ORDER, ordered=True)
summary.sort_values(["mineral", "model"])


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
for model in MODEL_ORDER:
    sub = summary[summary["model"] == model].sort_values("mineral")
    if sub.empty:
        continue
    axes[0].plot(sub["mineral_label"], sub["mean_oh_distance_valid_a"], marker="o", linewidth=1.8, label=MODEL_LABELS[model], color=MODEL_COLORS[model])
    axes[1].plot(sub["mineral_label"], sub["n_detached_pairs"], marker="s", linewidth=1.8, label=MODEL_LABELS[model], color=MODEL_COLORS[model])
axes[0].set_ylabel("Mean tracked O-H distance, last 500 frames (Å)")
axes[1].set_ylabel("Detached O-H pairs, last 500 frames")
for ax in axes:
    ax.set_xlabel("Mineral")
    ax.tick_params(axis="x", rotation=20)
    ax.grid(alpha=0.25)
axes[0].legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()


## O-H Distance Distributions

In [ ]:
import re

times = pd.read_csv(time_path)
times["mineral"] = times["structure"].str.replace("_1layer_stable_surface_NBO_H", "", regex=False)
times["mineral_label"] = times["mineral"].map(MINERAL_LABELS)
valid = times[~times["detached"]].copy()
minerals = ["anorthite", "fayalite", "forsterite", "ilmenite"]

# 1. Dictionary mapping lowercase keys to clean "Formula\n(Name)" strings
CUSTOM_TITLES = {
    "anorthite": r"$\mathregular{CaAl_2SiO_4}$" + " (Anorthite)",
    "fayalite": r"$\mathregular{Fe_2SiO_4}$" + " (Fayalite)",
    "forsterite": r"$\mathregular{Mg_2SiO_4}$" + " (Forsterite)",
    "ilmenite": r"$\mathregular{FeTiO_3}$" + " (Ilmenite)"
}

fig, axes = plt.subplots(2, 2, figsize=(16, 9), sharex=True)
axes = axes.ravel()
x_smooth = np.linspace(0.85, 1.15, 400)

for ax, mineral in zip(axes, minerals):
    sub = valid[valid["mineral"] == mineral]
    for model in MODEL_ORDER:
        vals = sub.loc[sub["model"] == model, "oh_distance_a"].dropna().to_numpy()
        if len(vals) < 5:
            continue
        kde = gaussian_kde(vals, bw_method=0.15)
        y = kde.evaluate(x_smooth)
        ax.plot(x_smooth, y, lw=3.0, ls=MODEL_LS[model], label=MODEL_LABELS[model], color=MODEL_COLORS[model])
        ax.fill_between(x_smooth, y, color=MODEL_COLORS[model], alpha=0.08)
    
    # 2. Use the new custom dictionary for the titles instead of MINERAL_LABELS
    ax.set_title(CUSTOM_TITLES.get(mineral, mineral), fontsize=16)
    ax.set_ylabel("Density", fontsize=16)
    ax.grid(alpha=0.15, linestyle=":")
    
    # Update tick font sizes (set both x and y to 16 for consistency, or just x)
    ax.tick_params(axis='x', labelsize=16)
    ax.tick_params(axis='y', labelsize=14) # Optional: slightly larger y-ticks to match
    
    # Set the ymin to 0 for the current subplot dynamically
    ax.set_ylim(bottom=0)

# Correctly apply x-labels and x-limits to the bottom row subplots
for ax in axes[-2:]:
    ax.set_xlabel("O-H distance (Å)", fontsize=16)
    ax.set_xlim(left=0.5, right=1.5)

axes[0].legend(frameon=False, fontsize=12)
fig.tight_layout()
# plt.show()
plt.savefig('FM_Lunar_OH.png', dpi=300, bbox_inches='tight')

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.legend as lg
import matplotlib.ticker as ticker
from scipy import interpolate
from matplotlib.ticker import LogFormatterSciNotation 
import matplotlib as mpl
import os, sys
import pandas as pd 
import matplotlib.style
import matplotlib.patches as patches


import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times", "Nimbus Roman", "DejaVu Serif"],
})


mpl.rcParams['font.size'] = 23
mpl.rcParams["font.weight"] = "bold"
# mpl.rcParams['font.family'] = 'Georgia'
mpl.rcParams['legend.fontsize'] = 23
mpl.rcParams['axes.linewidth'] = 3.0
mpl.rcParams['lines.linewidth'] = 3.0
mpl.rcParams['axes.labelweight'] = "bold"
mpl.rcParams['axes.titleweight'] = "bold"
mpl.rcParams['xtick.major.size'] = 8
mpl.rcParams['xtick.top']=True
mpl.rcParams['ytick.right']=True
mpl.rcParams['xtick.minor.size'] = 4
mpl.rcParams['xtick.major.width'] = 2
mpl.rcParams['xtick.minor.width'] = 1.5
mpl.rcParams['xtick.direction'] = "in"
mpl.rcParams['ytick.major.size'] = 8
mpl.rcParams['ytick.minor.size'] = 4
mpl.rcParams['ytick.major.width'] = 2
mpl.rcParams['ytick.minor.width'] = 1
mpl.rcParams['ytick.direction'] = "in"
mpl.rcParams['xtick.major.top'] = True
mpl.rcParams['xtick.minor.top'] = True
mpl.rcParams['ytick.major.right'] = True
mpl.rcParams['ytick.minor.right'] = True

## Pair-Level Diagnostics

In [ ]:

pairs = pd.read_csv(pair_path)
pairs["mineral"] = pairs["structure"].str.replace("_1layer_stable_surface_NBO_H", "", regex=False)
pairs["mineral_label"] = pairs["mineral"].map(MINERAL_LABELS)
pairs["model_label"] = pairs["model"].map(MODEL_LABELS)
pairs.sort_values(["detached_fraction", "max_oh_distance_a"], ascending=False).head(30)


## Hydroxylated Benchmark Timing and Memory

In [ ]:

perf_summary = pd.read_csv(perf_summary_path).copy()
perf_summary["model"] = perf_summary["model"].replace({"mace_mh1": "mace_mh"})
perf_summary["model_label"] = perf_summary["model"].map(MODEL_LABELS)
perf_summary["total_time_min"] = perf_summary["total_time_s"] / 60.0
perf_summary.sort_values("mean_time_per_step_ms")


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
plot_df = perf_summary.sort_values("mean_time_per_step_ms")
x = np.arange(len(plot_df))
colors = [MODEL_COLORS[m] for m in plot_df["model"]]
axes[0].bar(x, plot_df["total_time_min"], color=colors, edgecolor="white")
axes[0].set_xticks(x)
axes[0].set_xticklabels(plot_df["model_label"], rotation=25, ha="right")
axes[0].set_ylabel("Total time for four OH runs (min)")
axes[0].grid(axis="y", alpha=0.25)
axes[1].bar(x, plot_df["max_peak_gpu_mb"] / 1024, color=colors, edgecolor="white")
axes[1].set_xticks(x)
axes[1].set_xticklabels(plot_df["model_label"], rotation=25, ha="right")
axes[1].set_ylabel("Peak GPU memory (GB)")
axes[1].grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()
